
![image_1781276125181.png](./image_1781276125181.png "image_1781276125181.png")


**Get customer details based on their email**: Let's add a function to retrieve a customer detail based on their email address


In [0]:
%sql
CREATE OR REPLACE FUNCTION dbx_apps_poc.rag.get_customer_by_email(
    email_input STRING COMMENT 'Customer email used to retrieve customer information'
) 
RETURNS TABLE (
    customerID BIGINT,
    first_name STRING,
    last_name STRING,
    email_address STRING,
    phone_number STRING,
    address STRING,
    city STRING,
    state STRING,
    postal_zip_code STRING,
    country STRING,
    continent STRING,
    gender STRING
) 
COMMENT 'Returns the customer record matching the provided email address. Includes its customerID, first_name, last_name and more'
RETURN (
    SELECT 
        * 
    FROM samples.bakehouse.sales_customers 
    WHERE email_address = email_input
    LIMIT 1
);

In [0]:
%sql
SELECT * FROM dbx_apps_poc.rag.get_customer_by_email('brittanyramos@example.org')

In [0]:
%pip install langgraph --upgrade

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import random
import mlflow
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# Auto logging and tracing 
mlflow.langchain.autolog()

class State(MessagesState):
    pass

llm = ChatDatabricks(endpoint='databricks-gpt-oss-120b')

uc_tools_names = ("dbx_apps_poc.rag.*",)

# Define the UC hosted functions/tools
tools = UCFunctionToolkit(function_names=list(uc_tools_names)).tools

llm_with_tools = llm.bind_tools(tools)

def tool_calling_llm(state: State):
    return {
        "messages": [llm_with_tools.invoke(state["messages"])]
    }


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)

graph = builder.compile()

result = graph.invoke({
    "messages": [HumanMessage("Can you tell me more info about brittanyramos@example.org")]
})
for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")



In [0]:
result = graph.invoke({"messages": [HumanMessage("Which table are you using to get the user details")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")

In [0]:
result = graph.invoke({"messages": [HumanMessage("Return details for the Red Graphics Card")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")